# Seed the customer project

This notebook fills the customer PostHog project with 14 days of events for Tidewater, a fictional ticketing company, including six planted problems (P1 to P6).

**This cell** imports what the notebook needs and loads `POSTHOG_INGEST_HOST` and `CUSTOMER_PROJECT_TOKEN` from the `.env` file in the parent folder.

In [1]:
import json
import os
import random
import uuid
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

POSTHOG_INGEST_HOST = os.environ["POSTHOG_INGEST_HOST"].rstrip("/")
CUSTOMER_PROJECT_TOKEN = os.environ["CUSTOMER_PROJECT_TOKEN"]

## Send helper

`send_batch(events)` POSTs events to `{POSTHOG_INGEST_HOST}/batch/` in chunks of 500 and prints the status code of each chunk. `historical_migration` is on because most of these events are backdated.

In [2]:
CHUNK_SIZE = 500


def send_batch(events):
    for start in range(0, len(events), CHUNK_SIZE):
        chunk = events[start : start + CHUNK_SIZE]
        response = requests.post(
            f"{POSTHOG_INGEST_HOST}/batch/",
            json={
                "api_key": CUSTOMER_PROJECT_TOKEN,
                "historical_migration": True,
                "batch": chunk,
            },
            timeout=60,
        )
        print(f"chunk {start // CHUNK_SIZE + 1}: {response.status_code}")

## Shared setup

`random.seed(42)` makes the IDs, amounts, counts and time offsets repeat when you run the whole notebook from the top. Absolute timestamps still move, because the window always ends at the current time.

The helpers split the 14 days into 24-hour windows counted back from now, so the last window is exactly the last 24 hours. Random UUIDs come from the seeded generator too, so they also repeat.

In [3]:
random.seed(42)

NOW = datetime.now(timezone.utc)
START = NOW - timedelta(days=14)
DAY_WINDOWS = [(START + timedelta(days=i), START + timedelta(days=i + 1)) for i in range(14)]

USERS = [f"user_{i:03d}" for i in range(1, 61)]

JS_LIB = {"$lib": "posthog-js", "$lib_version": "1.444.4"}
PYTHON_LIB = {"$lib": "posthog-python", "$lib_version": "7.82.2"}


def about(n):
    return random.randint(round(n * 0.85), round(n * 1.15))


def random_timestamp(window_start, window_end):
    seconds = random.uniform(0, (window_end - window_start).total_seconds())
    return (window_start + timedelta(seconds=seconds)).isoformat()


def random_uuid():
    return str(uuid.UUID(int=random.getrandbits(128), version=4))


def make_event(event, distinct_id, timestamp, properties):
    return {"event": event, "distinct_id": distinct_id, "timestamp": timestamp, "properties": properties}

## Backend events (P6)

Each of the 60 users gets one `$identify` event with no `$set` properties, and about 40 `order_completed` events are generated per day.

**Planted problem P6:** every `order_completed` is sent twice with the same timestamp and `order_id`. The events carry no `uuid`, so PostHog gives each copy its own and stores both.

In [4]:
backend_events = [
    make_event("$identify", user, random_timestamp(START, START + timedelta(hours=1)), dict(PYTHON_LIB))
    for user in USERS
]

order_number = 0
for window_start, window_end in DAY_WINDOWS:
    for _ in range(about(40)):
        order_number += 1
        order = make_event(
            "order_completed",
            random.choice(USERS),
            random_timestamp(window_start, window_end),
            {
                **PYTHON_LIB,
                "order_id": f"ord_{order_number:06d}",
                "amount": round(random.uniform(15, 180), 2),
            },
        )
        backend_events += [order, order]

## Web events (P3)

About 250 `$pageview` events per day from `posthog-js` 1.444.4, across 6 URLs on `https://tidewater.example/`.

**Planted problem P3:** every visit uses a new random UUID as its `distinct_id` and is never identified, so web traffic can't be tied to a user.

In [5]:
WEB_URLS = [
    "https://tidewater.example/",
    "https://tidewater.example/events",
    "https://tidewater.example/events/harbor-lights",
    "https://tidewater.example/checkout",
    "https://tidewater.example/account",
    "https://tidewater.example/help",
]

web_events = [
    make_event(
        "$pageview",
        random_uuid(),
        random_timestamp(window_start, window_end),
        {**JS_LIB, "$current_url": random.choice(WEB_URLS)},
    )
    for window_start, window_end in DAY_WINDOWS
    for _ in range(about(250))
]

## Legacy site events (P5)

About 60 `$pageview` events per day from a second site.

**Planted problem P5:** this site runs the very old `posthog-js` 1.36.0, and its events have no `$current_url`. Each event uses a random UUID as its `distinct_id`.

In [6]:
legacy_events = [
    make_event(
        "$pageview",
        random_uuid(),
        random_timestamp(window_start, window_end),
        {"$lib": "posthog-js", "$lib_version": "1.36.0"},
    )
    for window_start, window_end in DAY_WINDOWS
    for _ in range(about(60))
]

## Mobile events (P1)

About 120 `$screen` events per day from `posthog-react-native` 4.85.1, sent by the 60 users.

**Planted problem P1:** only the first 11 of the 14 windows get events, so all mobile events stop exactly 3 days before now.

In [7]:
MOBILE_SCREENS = ["Home", "EventList", "EventDetail", "Checkout", "Tickets"]

mobile_events = [
    make_event(
        "$screen",
        random.choice(USERS),
        random_timestamp(window_start, window_end),
        {
            "$lib": "posthog-react-native",
            "$lib_version": "4.85.1",
            "$screen_name": random.choice(MOBILE_SCREENS),
        },
    )
    for window_start, window_end in DAY_WINDOWS[:11]
    for _ in range(about(120))
]

## Feature flag events (P2)

About 80 `$feature_flag_called` events per day for the last 5 days, sent by the 60 users through the backend SDK.

**Planted problem P2:** `new-checkout` always returns `false`, so nobody ever gets the new experience.

In [8]:
flag_events = [
    make_event(
        "$feature_flag_called",
        random.choice(USERS),
        random_timestamp(window_start, window_end),
        {**PYTHON_LIB, "$feature_flag": "new-checkout", "$feature_flag_response": False},
    )
    for window_start, window_end in DAY_WINDOWS[-5:]
    for _ in range(about(80))
]

## Exception events (P4)

For the first 13 days, about 5 `$exception` events per day with `app_version=2.3.1` and mixed error types.

**Planted problem P4:** the last 24 hours bring about 300 copies of a single `TypeError`, all on `app_version=2.4.0`, which looks like a bad release. These events come from `posthog-js`, so each one uses a random UUID as its `distinct_id` to keep the web SDK unidentified, as in P3.

In [9]:
BASELINE_ERRORS = [
    {"type": "ReferenceError", "value": "analytics is not defined"},
    {"type": "NetworkError", "value": "Failed to fetch /api/seats"},
    {"type": "RangeError", "value": "Invalid time value"},
    {"type": "SyntaxError", "value": "Unexpected token '<' in JSON at position 0"},
]
RELEASE_ERROR = {"type": "TypeError", "value": "Cannot read properties of undefined (reading 'total')"}

exception_events = [
    make_event(
        "$exception",
        random_uuid(),
        random_timestamp(window_start, window_end),
        {**JS_LIB, "app_version": "2.3.1", "$exception_list": [random.choice(BASELINE_ERRORS)]},
    )
    for window_start, window_end in DAY_WINDOWS[:13]
    for _ in range(about(5))
]

exception_events += [
    make_event(
        "$exception",
        random_uuid(),
        random_timestamp(*DAY_WINDOWS[13]),
        {**JS_LIB, "app_version": "2.4.0", "$exception_list": [RELEASE_ERROR]},
    )
    for _ in range(about(300))
]

## Check before sending

Combines all the events in time order, prints a count by event name and `$lib`, and stops if the total reaches 25,000. Nothing has been sent yet.

In [10]:
events = sorted(
    backend_events + web_events + legacy_events + mobile_events + flag_events + exception_events,
    key=lambda e: e["timestamp"],
)

counts = Counter((e["event"], e["properties"]["$lib"]) for e in events)
for (event, lib), count in sorted(counts.items()):
    print(f"{event:<24}{lib:<24}{count:>6}")
print(f"{'TOTAL':<48}{len(events):>6}")

assert len(events) < 25_000

$exception              posthog-js                 352
$feature_flag_called    posthog-python             425
$identify               posthog-python              60
$pageview               posthog-js                4454
$screen                 posthog-react-native      1312
order_completed         posthog-python            1162
TOTAL                                             7765


## Send everything

Sends all the events to the customer project and prints each chunk's status code.

**Run this once.** PostHog keeps everything it receives, so running this cell again adds a second full copy of the data.

In [11]:
send_batch(events)

chunk 1: 200
chunk 2: 200
chunk 3: 200
chunk 4: 200
chunk 5: 200
chunk 6: 200
chunk 7: 200
chunk 8: 200
chunk 9: 200
chunk 10: 200
chunk 11: 200
chunk 12: 200
chunk 13: 200
chunk 14: 200
chunk 15: 200
chunk 16: 200


In [12]:
import os, requests
from dotenv import load_dotenv
load_dotenv("../.env")
r = requests.post(os.environ["POSTHOG_INGEST_HOST"] + "/i/v0/e/", json={
"api_key": os.environ["CUSTOMER_PROJECT_TOKEN"],
"event": "$identify",
"distinct_id": "contractor-placeholder",
"properties": {"$set": {"email": "ops@contractor-example.com"}},
})
print(r.status_code)

200
